# 00 · Configuración de Hugging Face Hub

**Objetivo:** configurar el acceso a Hugging Face Hub para subir y descargar modelos entrenados de forma reproducible.

**Contexto:** los modelos entrenados (.pt) pueden ser muy pesados (decenas o cientos de MB), por lo que no es buena práctica versionarlos en GitHub. En su lugar, se alojan en Hugging Face Hub, plataforma especializada en hosting de modelos de inteligencia artificial.

## 1. Instalación de la librería

La librería `huggingface_hub` permite interactuar con la plataforma desde Python.

In [1]:
import sys
!{sys.executable} -m pip install huggingface_hub --quiet
print("Librería instalada correctamente")

Librería instalada correctamente



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Introducción segura del token

El token se introduce con `getpass`, que oculta el texto mientras se escribe para que no quede visible en el notebook.

In [2]:
from getpass import getpass

token = getpass("Introduce tu token de Hugging Face: ")
print(f"Token recibido (longitud: {len(token)} caracteres)")

Token recibido (longitud: 37 caracteres)


## 3. Guardado del token como variable de entorno

El token se guarda como variable de entorno permanente del usuario en Windows, de modo que cualquier notebook futuro pueda acceder a él sin necesidad de volver a introducirlo.

In [3]:
import os
import subprocess

os.environ["HUGGINGFACE_TOKEN"] = token

subprocess.run(
    ["setx", "HUGGINGFACE_TOKEN", token],
    capture_output=True,
    text=True,
    check=True
)

print("Token guardado como variable de entorno permanente")
print("En futuros notebooks se accede con: os.environ['HUGGINGFACE_TOKEN']")

Token guardado como variable de entorno permanente
En futuros notebooks se accede con: os.environ['HUGGINGFACE_TOKEN']


## 4. Verificación de la conexión con Hugging Face

Se utiliza el token para verificar que la conexión funciona correctamente y obtener información del usuario autenticado.

In [4]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HUGGINGFACE_TOKEN"])
info = api.whoami()

print(f"Conexión verificada correctamente")
print(f"Usuario autenticado: {info['name']}")
print(f"Tipo de cuenta: {info.get('type', 'user')}")

C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Conexión verificada correctamente
Usuario autenticado: hugosanchezgallego
Tipo de cuenta: user


## 5. Subida del modelo entrenado

Se sube el modelo `best.pt` resultante del primer entrenamiento al repositorio `futbol-detector-v1` creado previamente en Hugging Face.

In [5]:
from huggingface_hub import upload_file
from pathlib import Path

REPO_ID = f"{info['name']}/futbol-detector-v1"
MODEL_PATH = Path("../../models/best.pt")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"No se encuentra el modelo en: {MODEL_PATH.resolve()}")

print(f"Tamaño del modelo: {MODEL_PATH.stat().st_size / (1024*1024):.2f} MB")
print(f"Subiendo a: {REPO_ID}")

url = upload_file(
    path_or_fileobj=str(MODEL_PATH),
    path_in_repo="best.pt",
    repo_id=REPO_ID,
    token=os.environ["HUGGINGFACE_TOKEN"],
    commit_message="Subida inicial del modelo v1 entrenado con 152 frames"
)

print(f"\nModelo subido correctamente")
print(f"URL: {url}")

Tamaño del modelo: 21.48 MB
Subiendo a: hugosanchezgallego/futbol-detector-v1


Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   2%|▏         |  527kB / 22.5MB, 50.8kB/s  
Processing Files (0 / 1):   5%|▍         | 1.05MB / 22.5MB,  100kB/s  
Processing Files (0 / 1):  16%|█▋        | 3.69MB / 22.5MB,  348kB/s  
Processing Files (0 / 1):  37%|███▋      | 8.43MB / 22.5MB,  788kB/s  
Processing Files (0 / 1):  75%|███████▍  | 16.9MB / 22.5MB, 1.56MB/s  
Processing Files (0 / 1):  98%|█████████▊| 22.1MB / 22.5MB, 2.01MB/s  
Processing Files (1 / 1): 100%|██████████| 22.5MB / 22.5MB, 1.96MB/s  
Processing Files (1 / 1): 100%|██████████| 22.5MB / 22.5MB, 1.96MB/s  
New Data Upload: 100%|██████████| 22.5MB / 22.5MB, 1.96MB/s  



Modelo subido correctamente
URL: https://huggingface.co/hugosanchezgallego/futbol-detector-v1/commit/ce1bcee532b4fe95b28f73bf5fd445bc1003bc2a


## 6. Creación del README del repositorio del modelo

Se sube un README al repositorio de Hugging Face documentando el modelo, sus métricas y cómo utilizarlo.

In [6]:
readme_content = f"""---
license: mit
tags:
- yolov8
- object-detection
- football
- soccer
- sports-analytics
---

# Futbol Detector v1

Modelo YOLOv8s entrenado para detectar jugadores, árbitros y balón en vídeos de fútbol profesional.

## Contexto

Este modelo forma parte del Trabajo de Fin de Grado titulado *"Diseño de un sistema de asistencia al análisis táctico deportivo basado en procesamiento de imagen"*, desarrollado en el Grado en Ingeniería Informática de la Universidad Alfonso X el Sabio.

## Dataset de entrenamiento

- **Fuente:** DFL Bundesliga Data Shootout (Kaggle)
- **Imágenes etiquetadas:** 152 (v1)
- **Clases:** ball, player, referee
- **División:** 85% entrenamiento, 15% validación

## Resultados de validación

| Clase | Precisión | Recall | mAP50 |
|---|---|---|---|
| Jugador | 0.885 | 0.974 | 0.949 |
| Árbitro | 0.936 | 0.977 | 0.962 |
| Balón | 0.909 | 0.500 | 0.549 |
| Promedio | 0.910 | 0.817 | 0.820 |

## Uso

```python
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="{REPO_ID}", filename="best.pt")
model = YOLO(model_path)

results = model("ruta/al/video.mp4")
```

## Repositorio del proyecto

[https://github.com/HugoSanchezGallego/TFG](https://github.com/HugoSanchezGallego/TFG)
"""

api.upload_file(
    path_or_fileobj=readme_content.encode(),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    commit_message="Añadir README con métricas y documentación"
)

print(f"README subido correctamente")
print(f"Página del modelo: https://huggingface.co/{REPO_ID}")

README subido correctamente
Página del modelo: https://huggingface.co/hugosanchezgallego/futbol-detector-v1


## 7. Conclusiones

Tras este notebook:

- El token de Hugging Face queda guardado de forma segura como variable de entorno del usuario.
- El modelo `best.pt` queda alojado en Hugging Face Hub, accesible públicamente.
- El repositorio del modelo cuenta con un README profesional que documenta el contexto, métricas y uso.

A partir de ahora, cualquier notebook puede cargar el modelo directamente desde Hugging Face sin necesidad de tenerlo en local, lo que facilita la reproducibilidad del trabajo.